### 메타데이터 필터링 - 조건을 걸어서 검색
- 작년 경제 기사중에서만 찾아줘
- 중요한 뉴스만 찾아줘
- 어떤 기자가 쓴 뉴스만 찾아줘

In [21]:
import chromadb
import os
from chromadb.utils.embedding_functions import OpenAIEmbeddingFunction  # chromaDB 에서 Openai 임베딩 모델
from dotenv import load_dotenv
from openai import OpenAI
import pandas as pd

load_dotenv()   # .env 파일에 저장되어있는 api 키 가져오기
client = OpenAI()

In [22]:
embeddings = OpenAIEmbeddingFunction(
    model_name="text-embedding-3-small"
)

load_client = chromadb.PersistentClient(path="news_chroma_db")
load_collection = load_client.get_collection(
    name="news_collection",
    embedding_function=embeddings   # 임베딩 모델을 뭘 썻는지 알려줘야해서 
)

In [4]:
question = "주식이 언제 오를까요?"

result = load_collection.query(#컬렉션에 접근 후 검색 요청,  질문을 전달하는 함수
                                query_texts=[question], # 질문저장을 리스트로 한다
                                                        # 내부적으로 임베딩 벡터 변환
                                                        # 저장된 문서 벡터 코사인유사도 비교
                                                        # 가까운 문서 5개 반환
                                n_results=5) #결과 5가지
result

{'ids': [[]],
 'embeddings': None,
 'documents': [[]],
 'uris': None,
 'included': ['metadatas', 'documents', 'distances'],
 'data': None,
 'metadatas': [[]],
 'distances': [[]]}

In [5]:
result_filtered = load_collection.query(query_texts=[question], n_results=3,
                                        where={"카테고리" : {"$eq" : "IT과학"}}
                                        )

In [6]:
result_filtered

{'ids': [[]],
 'embeddings': None,
 'documents': [[]],
 'uris': None,
 'included': ['metadatas', 'documents', 'distances'],
 'data': None,
 'metadatas': [[]],
 'distances': [[]]}

In [7]:
result_filtered = load_collection.query(query_texts=["question"], n_results=3,
                                        where_document={"$contains" : "홈쇼핑"}
                                        )
result_filtered

{'ids': [[]],
 'embeddings': None,
 'documents': [[]],
 'uris': None,
 'included': ['metadatas', 'documents', 'distances'],
 'data': None,
 'metadatas': [[]],
 'distances': [[]]}

# 1. 데이터 구축영역

- 데이터 불러오기(데이터 로드) - 문서 적재
- 데이터 전처리 -  텍스트 추출, 청킹
- 임베딩 생성   -  텍스트에서 벡터화
- 메타데이터 생성 - 문서명, 날짜 , 문서명, 출처
 

### 허깅 페이스데이터 셋
-  https://huggingface.co/datasets/klue/klue

In [8]:
from datasets import load_dataset

In [9]:
dataset = load_dataset(
    "klue/klue", #"klue" 형식은 맞지 않고 HfUriError -> 저장소 ID - namespace/name를 사용
    "mrc",
    split="train"
)

df = dataset.to_pandas()
df.head()


,title,context,news_category,source,guid,is_impossible,question_type,question,answers
0,제주도 장마 시작 … 중부는 이달 말부터,올여름 장마가 17일 제주도에서 시작됐다. 서울 등 중부지방은 예년보다 사나흘 정도...,종합,hankyung,klue-mrc-v1_train_12759,False,1,북태평양 기단과 오호츠크해 기단이 만나 국내에 머무르는 기간은?,"{'answer_start': [478, 478], 'text': ['한 달가량',..."
1,"부산정보산업진흥원, 과기부 지역SW서비스사업화 지원사업 4개 과제 선정",부산시와 (재)부산정보산업진흥원(원장 이인숙)이 ‘2020~2021년 지역SW서비스...,경제,acrofan,klue-mrc-v1_train_06118,True,3,성공적인 성과를 보인 지역SW서비스사업화 지원사업의 주최자는?,"{'answer_start': [18, 21, 1673], 'text': ['원장 ..."
2,"부산정보산업진흥원, 과기부 지역SW서비스사업화 지원사업 4개 과제 선정",부산시와 (재)부산정보산업진흥원(원장 이인숙)이 ‘2020~2021년 지역SW서비스...,경제,acrofan,klue-mrc-v1_train_14934,False,2,지능형 생산자동화 기반기술을 개발중인 스타트업은?,"{'answer_start': [1422], 'text': ['삼보테크놀로지']}"
3,로버트 헨리 딕,"미국 세인트루이스에서 태어났고, 프린스턴 대학교에서 학사 학위를 마치고 1939년에...",NaN,wikipedia,klue-mrc-v1_train_02900,True,3,로버트 헨리 딕이 1946년에 매사추세츠 연구소에서 개발한 것은 무엇인가?,"{'answer_start': [127], 'text': ['레이다']}"
4,나루세 요시히사,시범 경기에서는 16이닝을 던져 15실점을 기록하는 등 성적이 좋지 않았지만 본인으...,NaN,wikipedia,klue-mrc-v1_train_02773,False,2,개막전에서 3안타 2실점을 기록해서 패한 선수는?,"{'answer_start': [107], 'text': ['와쿠이 히데아키']}"


In [10]:
#df.head() # 앞에 5개 행
# df.shape #컬럼 및 컬럼 값 개수 
df.columns.tolist() # 리스트에 컬럼 담기 -> 세로 , print -> 가로방향


['title',
 'context',
 'news_category',
 'source',
 'guid',
 'is_impossible',
 'question_type',
 'question',
 'answers']

In [11]:
data = df.head(300).copy() # 데이터 300개만 사용
docs = data["context"].tolist() # 컨텍트 컬럼을 리스트에 저장
ids = data["guid"].tolist() # 가이드 컬럼을 리스트에 저장

metadata = data[ # 메타 데이터는 실 데이터에서 컬럼 5개 지정하여 리스트에 저장 
    [
        "title",
        "news_category",
        "source",
        "question",
        "is_impossible"
    ]
].fillna("").to_dict("records") # 결측치를 공백으로 채우고, 레코드는 각 행을 변환시키는 조건 -> 딕셔너리

# print(len(docs)) # 몇개인지
# print(docs[0][:300])# 300개 
print(metadata[54])# 메타 데이터 0번


{'title': '年수익률 5~6% … 예금의 2배...오피스텔·상가로 몰리는 투자자', 'news_category': '부동산', 'source': 'hankyung', 'question': '상가보다 낮은 투자수익률을 보인 수익형 부동산의 연 투자수익률은?', 'is_impossible': False}


In [12]:
## 3. 임베딩 함수 준비
import chromadb # uv add chromadb
from chromadb.utils.embedding_functions import OpenAIEmbeddingFunction
# openai 임베딩 API를 사용 -> chromadb와 통신되야한다 - 임베딩 함수 클래스를 가져옴

embeddings = OpenAIEmbeddingFunction( # 모델명을 가진 파라미터로 사용할 임베딩 모델 지정 후 설정을 가진 임베딩 도구 생성
    model_name="text-embedding-3-small"
)

In [13]:
## 4. 별도 ChromaDB 컬렉션 생성

# 기존 뉴스 컬렉션과 섞이지 않도록 이름을 다르게
# 지정합니다.

chroma_client = chromadb.PersistentClient( # 디스크 저장기능 수행 - 폴더 생성 및 연결
    path="./klue_mrc_chroma_db" # DB 연결 정보 
)

klue_collection = chroma_client.get_or_create_collection( # 해당 DB 경로에 접근하여 컬렉션 가져오기 또는 생성
    name="klue_mrc_collection", # 컬렉션 이름 
    embedding_function=embeddings, # 임베딩함수에서 사용할 벡터 지정
    configuration={ # 검색방식은 코사인 유사도 설정
        "hnsw": {
            "space": "cosine"
        }
    }
)

print(klue_collection.name)
print(klue_collection.count())

klue_mrc_collection
0


In [14]:
## 5. 문서 저장
klue_collection.upsert(  # 컬렉션 생성 후 upsert 함수 사용하여 문서 적재 -> insert + update 
                         # ID가 없으면 새문서 추가 또는 기존 문서 갱신
    ids=ids,# 고유 번호
    documents=docs,# 벡더로 변환될 실제 문서 본문
    metadatas=metadata # 문서에대한 부가정보
)

print("저장된 문서 수:", klue_collection.count())

저장된 문서 수: 300


In [24]:
## 6. 질문 검색

# question = "장마는 한반도에 얼마나 오래 머무르나요?"
question = "왜 강아지는 하늘을 날아다녀요? "

result = klue_collection.query( #클루 컬렉션에 질문
    query_texts=[question], # 질문은 리스트에
    n_results=3 # 가장 가까운 3가지결과
)

for i, doc in enumerate(result["documents"][0]): #결과에 문서0번 컬럼에 대한 질문으로 순회
    print(f"[검색 결과 {i + 1}]") # 순회시 i가 올라간다
    print(doc[:500]) # 각 문서의 500자만 출력
    print()



for metadata_item in result["metadatas"][0]: # 검색 결과의 메타데이터도 확인합니다.
    print(metadata_item)


[검색 결과 1]
“아폴로우주선을 타고 지금까지 여섯 차례에 걸쳐 12명이 달 표면에 발을 디뎠습니다. 그럼 가장 깊은 바다라고 하는 마리아나 해구의 챌린저 해연(깊이 1만911±40ｍ)까지 내려간 사람은 몇 명일까요? 세 명입니다. 그것도 1960년과 2012년 두 차례뿐이죠.” 지난 18일 경기 안산시에 있는 한국해양과학기술원에서 만난 강정극 원장(사진)은 “우주에 비해 바다는 우리에게 아주 가깝고 친숙하지만 우리가 바다에 대해 아는 것은 그리 많지 않다”며 “기후 변화, 미래 식량, 광물 자원, 에너지 등의 문제는 모두 바다를 통해 해결할 수 있기 때문에 바다를 연구하는 것은 매우 중요한 일”이라고 강조했다. 한국해양과학기술원이 올해로 설립 40주년을 맞았다. 1973년 한국과학기술연구원(KIST) 부설 해양개발연구소로 설립돼 1990년 ‘한국해양연구소’로 독립했고, ‘한국해양연구원’을 거쳐 작년 7월 ‘한국해양과학기술원’으로 이름이 바뀌었다. 강 원장은 “1960년대에 존 F 케네디 미국 대통

[검색 결과 2]
“아폴로우주선을 타고 지금까지 여섯 차례에 걸쳐 12명이 달 표면에 발을 디뎠습니다. 그럼 가장 깊은 바다라고 하는 마리아나 해구의 챌린저 해연(깊이 1만911±40ｍ)까지 내려간 사람은 몇 명일까요? 세 명입니다. 그것도 1960년과 2012년 두 차례뿐이죠.” 지난 18일 경기 안산시에 있는 한국해양과학기술원에서 만난 강정극 원장(사진)은 “우주에 비해 바다는 우리에게 아주 가깝고 친숙하지만 우리가 바다에 대해 아는 것은 그리 많지 않다”며 “기후 변화, 미래 식량, 광물 자원, 에너지 등의 문제는 모두 바다를 통해 해결할 수 있기 때문에 바다를 연구하는 것은 매우 중요한 일”이라고 강조했다. 한국해양과학기술원이 올해로 설립 40주년을 맞았다. 1973년 한국과학기술연구원(KIST) 부설 해양개발연구소로 설립돼 1990년 ‘한국해양연구소’로 독립했고, ‘한국해양연구원’을 거쳐 작년 7월 ‘한국해양과학기술원’으로 이름이 바뀌었다. 강 원장은 

In [25]:
## 7. 실제 질문과 정답 비교

# 데이터에 있는 질문을 그대로 검색해보세요.

test_question = data.iloc[0]["question"] #데이터에 첫번째 행에서 질문 컬럼값을 가져옴
real_answer = data.iloc[0]["answers"] # 데이터의 첫번쨰행에서 답변값을 가져옴 

print("질문:", test_question)
print("실제 정답:", real_answer)

# 그리고 검색합니다.

result = klue_collection.query( #클루컬렉션에 질문
    query_texts=[test_question],#질문을 리스트에 담음 
    n_results=3 # 가장 가까운 3가지 
)

print(result["documents"][0][0][:1000]) # 검색된 문서 전체에, 첫번째 질문결과, 가장가까운 문서, 앞에서부터 1000자

질문: 북태평양 기단과 오호츠크해 기단이 만나 국내에 머무르는 기간은?
실제 정답: {'answer_start': array([478, 478], dtype=int32), 'text': array(['한 달가량', '한 달'], dtype=object)}
주요 그룹 총수들은 다음주 설 연휴 동안 대부분 자택에 머물며 경영 구상에 몰두한다.13일 재계에 따르면 정몽구 현대자동차그룹 회장은 설 연휴 동안 서울 한남동 자택에서 경영 전략을 가다듬을 것으로 알려졌다. 최근 정 회장은 서울 삼성동 한국전력 본사에 건설할 글로벌비즈니스센터(GBC)를 조기에 착공할 수 있도록 고삐를 죄고 있다. 양력설을 쇠는 정 회장은 매년 음력설엔 특별한 일정 없이 휴식을 취해왔다.구본무 LG그룹 회장도 연휴 기간에 서울 한남동 자택에서 경영 구상을 할 예정이다. 올해로 취임 20주년을 맞은 구 회장은 앞서 “올해 사업 환경은 여전히 어려워 보이지만 기필코 시장을 선도하겠다는 굳은 각오로 방법을 찾고 힘을 모아 철저하게 실행해야 한다”고 강조했다.허창수 GS그룹 회장은 서울 이촌동 자택에서 차례를 지낸 뒤 가족과 휴식을 취하며 경영 구상에 전념할 계획이다. 지난 10일 전국경제인연합회 회장에 재선임된 허 회장은 3기 체제를 맞아 전경련 이미지 쇄신 방안도 고심할 것으로 전해졌다. 김승연 한화그룹 회장은 서울 가회동 자택에서 가족과 차례를 지내며 휴식을 취한다. 지난해 말 경영에 복귀한 만큼 삼성의 방산·화학부문 4개 계열사를 인수한 뒤 그룹 경쟁력 강화 방안을 숙고할 것으로 알려졌다. 권오준 포스코 회장과 이웅열 코오롱 회장도 자택에 머물며 가족과 함께 설을 보낼 예정이다.


다음 세 가지를 비교해보세요.

1. 사용자가 입력한 질문
2. 벡터 DB에서 검색된 context
3. 데이터셋에 기록된 answers


In [17]:
## 8. 메타데이터 필터링

# 예를 들어 뉴스 카테고리가 종합인 문서만 검색할
# 수 있습니다.

filtered_result = klue_collection.query( 
    query_texts=[question],
    n_results=3,
    where={
        "news_category": {
            "$eq": "종합" # equal 같다
        }
    }
)

filtered_result["documents"]

[['올여름 장마가 17일 제주도에서 시작됐다. 서울 등 중부지방은 예년보다 사나흘 정도 늦은 이달 말께 장마가 시작될 전망이다.17일 기상청에 따르면 제주도 남쪽 먼바다에 있는 장마전선의 영향으로 이날 제주도 산간 및 내륙지역에 호우주의보가 내려지면서 곳곳에 100㎜에 육박하는 많은 비가 내렸다. 제주의 장마는 평년보다 2~3일, 지난해보다는 하루 일찍 시작됐다. 장마는 고온다습한 북태평양 기단과 한랭 습윤한 오호츠크해 기단이 만나 형성되는 장마전선에서 내리는 비를 뜻한다.장마전선은 18일 제주도 먼 남쪽 해상으로 내려갔다가 20일께 다시 북상해 전남 남해안까지 영향을 줄 것으로 보인다. 이에 따라 20~21일 남부지방에도 예년보다 사흘 정도 장마가 일찍 찾아올 전망이다. 그러나 장마전선을 밀어올리는 북태평양 고기압 세력이 약해 서울 등 중부지방은 평년보다 사나흘가량 늦은 이달 말부터 장마가 시작될 것이라는 게 기상청의 설명이다. 장마전선은 이후 한 달가량 한반도 중남부를 오르내리며 곳곳에 비를 뿌릴 전망이다. 최근 30년간 평균치에 따르면 중부지방의 장마 시작일은 6월24~25일이었으며 장마기간은 32일, 강수일수는 17.2일이었다.기상청은 올해 장마기간의 평균 강수량이 350~400㎜로 평년과 비슷하거나 적을 것으로 내다봤다. 브라질 월드컵 한국과 러시아의 경기가 열리는 18일 오전 서울은 대체로 구름이 많이 끼지만 비는 오지 않을 것으로 예상돼 거리 응원에는 지장이 없을 전망이다.',
  '‘이정재 역조공’에 동양사태 피해자들 분노최근 ‘이정재 역조공’이 실시간 검색어 상위권에 등극. 배우 이정재가 팬에게 식사 대접을 했다는 보도자료 정리 기사가 쏟아졌기 때문. 이 바람에 이씨 시행사를 부당 지원한 동양그룹 대주주 일가를 (주)동양이 제소하려 한다는 기사가 묻혔으니.버거킹 그만두고 화장품업계를 흔드는 여자한국계 미국인 그레이스 최(30). 버거킹 입사 3개월 만에 그만두고 창업해 비비크림으로 돌풍을 일으킨 기업인. 올해는 혁신적인 색조화장품 제조기 ‘밍

In [ ]:
# 본문에 특정 단어가 포함된 문서만 검색하려면:

filtered_result = klue_collection.query( 
    query_texts=[question],
    n_results=3,
    where_document={
        "$contains": "제주" 
    }
)


In [19]:
# 벡터 DB 문서:
# context

# 메타데이터:
#   title
#   news_category
#   source
#   question
#   is_impossible

In [28]:
# ## 첫 검색 실습- 이미 정해진 컬럼
# question = df.loc[0, "question"] # 데이터 프래임 컬럼 위치 선택

# result = collection.query( # 컬렉션.질문 함수
#     query_texts=[question],
#     n_results=3
# )

# result["documents"]

In [29]:
## RAG용 컬럼 구성
docs = df["context"].tolist() 

ids = df["guid"].tolist()

metadata = df[
    [
        "title",
        "news_category",
        "source",
        "question",
        "is_impossible"
    ]
].fillna("").to_dict("records")



# 그다음 검색된 문서와 실제 정답을 비교합니다.

print("질문:", df.loc[0, "question"])
print("정답:", df.loc[0, "answers"])
print("검색 문서:", result["documents"][0][0])

## dp를 꼭 사용하고 싶다면

# dp도 벡터 검색 자체를 연습하는 데는 사용할 수 있습니다.

dataset = load_dataset(
    "klue/klue",
    "dp",
    split="train"
)

df = dataset.to_pandas()

docs = df["sentence"].tolist()
ids = df["sentence"].index.astype(str).tolist()

metadata = [
    {
        "word_count": len(row["word_form"]),
        "source": "KLUE-DP" #여기서 source는 해당 문서가 KLUE 의존 구문 분석 데이터에서 왔다는 표시입니
  다.
    }
    for _, row in df.iterrows()
]


질문: 북태평양 기단과 오호츠크해 기단이 만나 국내에 머무르는 기간은?
정답: {'answer_start': array([478, 478], dtype=int32), 'text': array(['한 달가량', '한 달'], dtype=object)}
검색 문서: 주요 그룹 총수들은 다음주 설 연휴 동안 대부분 자택에 머물며 경영 구상에 몰두한다.13일 재계에 따르면 정몽구 현대자동차그룹 회장은 설 연휴 동안 서울 한남동 자택에서 경영 전략을 가다듬을 것으로 알려졌다. 최근 정 회장은 서울 삼성동 한국전력 본사에 건설할 글로벌비즈니스센터(GBC)를 조기에 착공할 수 있도록 고삐를 죄고 있다. 양력설을 쇠는 정 회장은 매년 음력설엔 특별한 일정 없이 휴식을 취해왔다.구본무 LG그룹 회장도 연휴 기간에 서울 한남동 자택에서 경영 구상을 할 예정이다. 올해로 취임 20주년을 맞은 구 회장은 앞서 “올해 사업 환경은 여전히 어려워 보이지만 기필코 시장을 선도하겠다는 굳은 각오로 방법을 찾고 힘을 모아 철저하게 실행해야 한다”고 강조했다.허창수 GS그룹 회장은 서울 이촌동 자택에서 차례를 지낸 뒤 가족과 휴식을 취하며 경영 구상에 전념할 계획이다. 지난 10일 전국경제인연합회 회장에 재선임된 허 회장은 3기 체제를 맞아 전경련 이미지 쇄신 방안도 고심할 것으로 전해졌다. 김승연 한화그룹 회장은 서울 가회동 자택에서 가족과 차례를 지내며 휴식을 취한다. 지난해 말 경영에 복귀한 만큼 삼성의 방산·화학부문 4개 계열사를 인수한 뒤 그룹 경쟁력 강화 방안을 숙고할 것으로 알려졌다. 권오준 포스코 회장과 이웅열 코오롱 회장도 자택에 머물며 가족과 함께 설을 보낼 예정이다.


dp/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 3.12MB            

dp/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

c:\study-with-ai\.venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Lee\.cache\huggingface\hub\datasets--klue--klue. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


dp/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  618kB            

dp/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2000 [00:00<?, ? examples/s]